# FINS5557 Week 8 Case Study
## Quantitative ASX Equity Trading and Investment Platform Simulation

**Course:** FINS5557 Applied AI in Finance | UNSW Business School  
**Authors:** Hric & Lin, 2026

---

### Seminar Cross-Reference

This notebook implements two integrated pipelines from the Week 8 seminar:

| Pipeline | Description | Seminar Section |
|---|---|---|
| **A — Short-Term Trading** | Signal-driven ASX equity trading (RSI, MACD, Bollinger Bands + AI Analyst) | §2 The AI Trading Desk |
| **B — Long-Term Investment** | ESG-aware MVO portfolio for a superannuation mandate | §2 Portfolio AI Agent; §3 MVO vs. RL |

### DDF Pipeline Mapping (Hric & Lin, 2026)

| Station | Pipeline A | Pipeline B |
|---|---|---|
| **1 — ETL** | Synthetic ASX OHLCV data generation | Synthetic fundamentals + ESG scores |
| **2 — Feature Engineering** | RSI, MACD, Bollinger Bands | P/E, ROE, Debt/Equity, ESG score vector |
| **3 — Model Design** | AI Analyst signal (Gemini / mock) | MVO+ESG optimiser (Eq 8.5); HMM regime detection |
| **4 — Implementation** | Backtesting + performance metrics (SR, MDD, CR, AR) | Portfolio weights; ESG report; benchmark comparison |

### Gemini API Setup (Free — No Credit Card Required)

1. Get a free API key at **https://aistudio.google.com/apikey** — free tier: 15 RPM / 1M tokens per day
2. In Colab: click the **key icon** (Secrets) → `+ Add new secret`
   - Name: `GOOGLE_API_KEY` | Value: paste your key | Toggle **Notebook access** ON
3. Re-run the setup cell — the notebook auto-detects the secret
4. **No key?** The mock simulation runs automatically and produces representative output of identical structure

In [ ]:
# Auto-install required libraries if absent
import subprocess, sys

def _install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

try:
    import google.generativeai as genai
except ImportError:
    _install('google-generativeai')
    import google.generativeai as genai

try:
    import scipy
except ImportError:
    _install('scipy')

try:
    import hmmlearn
except ImportError:
    _install('hmmlearn')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── API key loading: Colab Secrets → env var → mock fallback ──
def _load_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('GOOGLE_API_KEY')
        if key:
            print('API key loaded from Colab Secrets.')
            return key
    except Exception:
        pass
    import os
    return os.getenv('GOOGLE_API_KEY')

GEMINI_API_KEY = _load_api_key()
USE_LIVE_API   = GEMINI_API_KEY is not None
GEMINI_MODEL   = 'gemini-1.5-flash'

if USE_LIVE_API:
    genai.configure(api_key=GEMINI_API_KEY)
    print(f'Live API enabled — model: {GEMINI_MODEL}')
else:
    print('No API key found — running full mock simulation.')
    print('Add GOOGLE_API_KEY to Colab Secrets to enable live Gemini commentary.')

np.random.seed(42)
print('Setup complete.')

---
## Pipeline A — Signal-Driven Short-Term ASX Equity Trading

### DDF Station 1: ETL — Synthetic ASX Market Data

The following cell generates synthetic OHLCV (Open, High, Low, Close, Volume) price series
for ten ASX-listed equities using Geometric Brownian Motion (GBM), calibrated to approximate
realistic return and volatility parameters for each sector.

**Educational note:** In a production system, this station would ingest live ASX data via the
ASX Market Data API or a vendor feed (Bloomberg, Refinitiv), with BCBS 239 data quality
controls applied at ingestion [Hric & Lin, 2026].

In [ ]:
# ── DDF Station 1: ETL — Synthetic ASX OHLCV Data ──

ASX_STOCKS = {
    'CBA.AX':  {'sector': 'Financials',  'mu': 0.10, 'sigma': 0.18, 'S0': 105.0},
    'BHP.AX':  {'sector': 'Materials',   'mu': 0.08, 'sigma': 0.25, 'S0': 45.0},
    'CSL.AX':  {'sector': 'Healthcare',  'mu': 0.12, 'sigma': 0.20, 'S0': 290.0},
    'RIO.AX':  {'sector': 'Materials',   'mu': 0.07, 'sigma': 0.22, 'S0': 120.0},
    'WES.AX':  {'sector': 'Consumer',    'mu': 0.09, 'sigma': 0.16, 'S0': 68.0},
    'NAB.AX':  {'sector': 'Financials',  'mu': 0.09, 'sigma': 0.17, 'S0': 38.0},
    'TLS.AX':  {'sector': 'Telecom',     'mu': 0.06, 'sigma': 0.14, 'S0': 4.2},
    'TCL.AX':  {'sector': 'Utilities',   'mu': 0.07, 'sigma': 0.15, 'S0': 14.5},
    'MQG.AX':  {'sector': 'Financials',  'mu': 0.11, 'sigma': 0.21, 'S0': 210.0},
    'FMG.AX':  {'sector': 'Materials',   'mu': 0.09, 'sigma': 0.30, 'S0': 22.0},
}

TRADING_DAYS = 252
dt = 1 / TRADING_DAYS
dates = pd.bdate_range(start='2023-01-02', periods=TRADING_DAYS)

def simulate_gbm(S0, mu, sigma, n, dt, seed=None):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n)
    log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
    prices = S0 * np.exp(np.cumsum(log_returns))
    return prices

# Build close price DataFrame (Station 1 output: clean price series)
price_data = {}
for i, (ticker, params) in enumerate(ASX_STOCKS.items()):
    price_data[ticker] = simulate_gbm(
        S0=params['S0'], mu=params['mu'], sigma=params['sigma'],
        n=TRADING_DAYS, dt=dt, seed=i
    )

prices = pd.DataFrame(price_data, index=dates)

print(f'Station 1 ETL complete: {len(prices)} trading days, {len(prices.columns)} ASX stocks')
print('Price series tail (AUD):')
print(prices.tail(3).round(2).to_string())

### DDF Station 2: Feature Engineering — Technical Indicators

Three technical indicators are computed for each stock. These constitute the signal surface
for the Analyst AI Agent in Station 3.

- **RSI (14-day):** Relative Strength Index — measures momentum; RSI > 70 signals overbought, RSI < 30 signals oversold
- **MACD (12/26/9):** Moving Average Convergence Divergence — crossover of MACD line above signal line is a buy signal
- **Bollinger Bands (20-day, 2σ):** price touching the upper band signals potential reversal; lower band signals potential oversold condition

In [ ]:
# ── DDF Station 2: Feature Engineering — Technical Indicators ──

def compute_rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast   = series.ewm(span=fast, adjust=False).mean()
    ema_slow   = series.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return macd_line, signal_line, histogram

def compute_bollinger(series, window=20, n_std=2):
    sma   = series.rolling(window).mean()
    std   = series.rolling(window).std()
    upper = sma + n_std * std
    lower = sma - n_std * std
    # %B: position within bands (0 = lower, 1 = upper)
    pct_b = (series - lower) / (upper - lower)
    return sma, upper, lower, pct_b

# Compute features for the focus stock: CBA.AX
focus = 'CBA.AX'
close = prices[focus]

rsi = compute_rsi(close)
macd_line, signal_line, macd_hist = compute_macd(close)
bb_sma, bb_upper, bb_lower, pct_b = compute_bollinger(close)

features = pd.DataFrame({
    'Close': close,
    'RSI_14': rsi,
    'MACD': macd_line,
    'MACD_Signal': signal_line,
    'MACD_Hist': macd_hist,
    'BB_Upper': bb_upper,
    'BB_Lower': bb_lower,
    'BB_PctB': pct_b,
}, index=dates).dropna()

# ── Visualise Station 2 outputs ──
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle(f'DDF Station 2: Technical Indicators — {focus}', fontsize=14, fontweight='bold')

# Price + Bollinger Bands
axes[0].plot(features.index, features['Close'], label='Close', color='#003366', linewidth=1.5)
axes[0].fill_between(features.index, features['BB_Upper'], features['BB_Lower'],
                     alpha=0.15, color='#C8102E', label='Bollinger Band (20d, 2σ)')
axes[0].plot(features.index, bb_sma.loc[features.index], '--', color='#C8102E', alpha=0.6, linewidth=1)
axes[0].set_ylabel('Price (AUD)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# RSI
axes[1].plot(features.index, features['RSI_14'], color='#003366', linewidth=1.5)
axes[1].axhline(70, color='#C8102E', linestyle='--', alpha=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.8, label='Oversold (30)')
axes[1].fill_between(features.index, 70, features['RSI_14'],
                     where=features['RSI_14'] > 70, alpha=0.2, color='#C8102E')
axes[1].fill_between(features.index, 30, features['RSI_14'],
                     where=features['RSI_14'] < 30, alpha=0.2, color='green')
axes[1].set_ylabel('RSI (14)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# MACD
axes[2].plot(features.index, features['MACD'], color='#003366', linewidth=1.5, label='MACD')
axes[2].plot(features.index, features['MACD_Signal'], color='#C8102E', linewidth=1.5, label='Signal')
axes[2].bar(features.index, features['MACD_Hist'],
            color=np.where(features['MACD_Hist'] >= 0, '#003366', '#C8102E'), alpha=0.4, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Station 2 complete: {len(features)} observation rows with RSI, MACD, and Bollinger features.')

### DDF Station 3: Model Design — AI Analyst Signal Generation

The Analyst AI Agent synthesises the technical indicator outputs into a trading signal
(BUY / HOLD / SELL) with a brief rationale. With the Gemini API, the agent generates
a professional equity report. Without a key, a complete mock simulation runs automatically,
producing output of identical structure.

This implements the **segment source classification** concept from [Fons et al., 2025]:
the mock report is annotated with DR (Direct Reference), FI (Financial Interpretation),
and EK (External Knowledge) segment types to illustrate the classification framework.

In [ ]:
# ── DDF Station 3a: AI Analyst Signal Generation (Gemini / Mock) ──

def build_analyst_prompt(ticker, latest):
    return f"""
You are a quantitative equity analyst generating a short trading signal report for {ticker}.
Analyse the following technical indicators and produce a structured report.

Latest readings (as of most recent trading day):
- Closing price: AUD {latest['Close']:.2f}
- RSI (14-day): {latest['RSI_14']:.1f}
- MACD: {latest['MACD']:.4f} | Signal line: {latest['MACD_Signal']:.4f} | Histogram: {latest['MACD_Hist']:.4f}
- Bollinger %B: {latest['BB_PctB']:.2f} (0=lower band, 1=upper band)
- Bollinger Upper: {latest['BB_Upper']:.2f} | Lower: {latest['BB_Lower']:.2f}

Produce a report with these sections:
1. SIGNAL: BUY, HOLD, or SELL (one word)
2. CONVICTION: HIGH, MEDIUM, or LOW
3. TECHNICAL ANALYSIS: 2-3 sentences interpreting the indicators (Direct Reference to data provided)
4. MARKET CONTEXT: 1-2 sentences on broader market conditions relevant to Australian financials
5. RISK FACTORS: 2-3 bullet points of key risks to the signal

Keep the total report under 200 words. Use professional analyst language.
""".strip()

def mock_analyst_report(ticker, latest):
    rsi = latest['RSI_14']
    macd_cross = 'positive' if latest['MACD_Hist'] > 0 else 'negative'
    pct_b = latest['BB_PctB']

    if rsi < 35:
        signal, conviction = 'BUY', 'MEDIUM'
    elif rsi > 65:
        signal, conviction = 'SELL', 'MEDIUM'
    elif macd_cross == 'positive' and pct_b < 0.6:
        signal, conviction = 'BUY', 'HIGH'
    else:
        signal, conviction = 'HOLD', 'LOW'

    report = f"""
SIGNAL: {signal}
CONVICTION: {conviction}

TECHNICAL ANALYSIS [DR]: The 14-day RSI reads {rsi:.1f}, placing the stock
{'in oversold territory below the 35 threshold' if rsi < 35 else
 'in overbought territory above the 65 threshold' if rsi > 65 else
 'in a neutral zone between 35 and 65'}.
The MACD histogram is {macd_cross} at {latest['MACD_Hist']:.4f}, indicating
{'bullish' if macd_cross == 'positive' else 'bearish'} momentum.
Bollinger %B at {pct_b:.2f} indicates the price is
{'near the upper band — potential resistance' if pct_b > 0.8 else
 'near the lower band — potential support' if pct_b < 0.2 else
 'positioned within the mid-band range, suggesting consolidation'}.

MARKET CONTEXT [EK]: Australian financial sector equities remain sensitive to
RBA monetary policy guidance; current cash rate settings affect net interest
margin trajectory for major banks. Regulatory capital requirements under APRA
Basel III finalisation continue to influence sector valuation multiples.

RISK FACTORS [FI]:
- Regime shift risk: strategy trained on low-volatility environment may misprice
  signals during elevated VIX periods (PSR monitoring recommended per Eq 8.6)
- Earnings surprise: quarterly results not yet incorporated in technical signals
- Liquidity risk: large-cap ASX financials generally liquid; mid-session volume
  concentration requires VWAP execution to minimise market impact

[Mock simulation — segment annotations: DR=Direct Reference, FI=Financial Interpretation, EK=External Knowledge]
""".strip()
    return signal, conviction, report

# Generate analyst report
latest = features.iloc[-1]

if USE_LIVE_API:
    try:
        client = genai.GenerativeModel(GEMINI_MODEL)
        prompt = build_analyst_prompt(focus, latest)
        response = client.generate_content(prompt)
        report_text = response.text
        # Parse signal from first line
        first_line = report_text.strip().split('\n')[0].upper()
        signal = 'BUY' if 'BUY' in first_line else 'SELL' if 'SELL' in first_line else 'HOLD'
        conviction = 'MEDIUM'
        print(f'Live Gemini report generated for {focus}.')
    except Exception as e:
        print(f'Gemini API error ({e}); falling back to mock.')
        signal, conviction, report_text = mock_analyst_report(focus, latest)
else:
    signal, conviction, report_text = mock_analyst_report(focus, latest)

print(f'\n{"="*60}')
print(f'AI ANALYST REPORT — {focus} — DDF Station 3')
print('='*60)
print(report_text)
print(f'\nFinal signal: {signal} | Conviction: {conviction}')

### DDF Station 4: Implementation — Backtesting and Performance Metrics

The signal generated by the AI Analyst is embedded within a simple rule-based strategy
that backtests over the full 252-day simulation period. The strategy applies a composite
signal score from RSI, MACD crossover, and Bollinger %B, then evaluates performance
against four metrics introduced in the Week 8 seminar:

- **Sharpe Ratio (SR)** — Eq 8.1: risk-adjusted return
- **Maximum Drawdown (MDD)** — peak-to-trough decline
- **Cumulative Return (CR)** — total return over the period
- **Annualised Return (AR)** — compound annual growth rate

In [ ]:
# ── DDF Station 4a: Backtesting — Signal-Driven Strategy ──

RBA_CASH_RATE = 0.0435  # RBA cash rate as of 2024 (4.35%)

def compute_signal_score(rsi, macd_hist, pct_b):
    score = 0.0
    # RSI component: oversold (+1) / overbought (-1)
    if rsi < 30:   score += 1.0
    elif rsi > 70: score -= 1.0
    else:          score += (50 - rsi) / 50  # linear gradient
    # MACD component: positive histogram = bullish
    score += np.sign(macd_hist) * 0.5
    # Bollinger component: lower band = buy signal, upper = sell
    score += (0.5 - pct_b)  # ranges from +0.5 (at lower) to -0.5 (at upper)
    return score

def backtest_strategy(prices_series, features_df, threshold_buy=0.5, threshold_sell=-0.3,
                      rf=RBA_CASH_RATE):
    scores = features_df.apply(
        lambda r: compute_signal_score(r['RSI_14'], r['MACD_Hist'], r['BB_PctB']), axis=1
    )
    # Position: 1=long, 0=cash, based on signal score
    position = pd.Series(0.0, index=scores.index)
    position[scores > threshold_buy]  = 1.0
    position[scores < threshold_sell] = 0.0

    daily_returns   = prices_series.loc[features_df.index].pct_change().shift(-1).fillna(0)
    strategy_returns = position * daily_returns

    # Benchmark: buy and hold
    bh_returns = daily_returns

    # ── Performance metrics ──
    def sharpe(rets, rf=rf):
        excess = rets.mean() * 252 - rf
        ann_vol = rets.std() * np.sqrt(252)
        return excess / ann_vol if ann_vol > 0 else 0.0

    def max_drawdown(rets):
        cum = (1 + rets).cumprod()
        roll_max = cum.cummax()
        drawdown = (cum - roll_max) / roll_max
        return drawdown.min()

    def cumulative_return(rets):
        return (1 + rets).prod() - 1

    def annualised_return(rets, periods=252):
        cr = cumulative_return(rets)
        n  = len(rets)
        return (1 + cr) ** (periods / n) - 1

    metrics = {
        'Strategy SR':   sharpe(strategy_returns),
        'Benchmark SR':  sharpe(bh_returns),
        'Strategy MDD':  max_drawdown(strategy_returns),
        'Benchmark MDD': max_drawdown(bh_returns),
        'Strategy CR':   cumulative_return(strategy_returns),
        'Benchmark CR':  cumulative_return(bh_returns),
        'Strategy AR':   annualised_return(strategy_returns),
        'Benchmark AR':  annualised_return(bh_returns),
    }
    return strategy_returns, bh_returns, position, scores, metrics

strat_rets, bh_rets, position, scores, metrics = backtest_strategy(prices[focus], features)

# ── Compute PSR: Performance Stability Ratio (Eq 8.6) ──
# Split into two halves: 'in-regime' (first half) and 'out-of-regime' (second half)
mid = len(strat_rets) // 2
sr_in  = (strat_rets.iloc[:mid].mean() * 252 - RBA_CASH_RATE) / (strat_rets.iloc[:mid].std() * np.sqrt(252))
sr_out = (strat_rets.iloc[mid:].mean() * 252 - RBA_CASH_RATE) / (strat_rets.iloc[mid:].std() * np.sqrt(252) + 1e-9)
psr = sr_out / sr_in if abs(sr_in) > 0.01 else float('nan')

print('Station 4 — Performance Metrics (Eq 8.1–8.2)')
print('='*52)
print(f'{"Metric":<25} {"Strategy":>10} {"Benchmark":>12}')
print('-'*52)
for k in ['SR', 'MDD', 'CR', 'AR']:
    sv = metrics[f'Strategy {k}']
    bv = metrics[f'Benchmark {k}']
    fmt = '.2f' if k == 'SR' else '.1%'
    fmt_w10 = f'>10{fmt}'   # build full format spec: e.g. '>10.2f' or '>10.1%'
    fmt_w12 = f'>12{fmt}'
    print(f'  {k:<23} {sv:{fmt_w10}} {bv:{fmt_w12}}')
print('-'*52)
print(f'  Performance Stability Ratio (Eq 8.6):   {psr:.3f}')
if not np.isnan(psr):
    if psr < 0:   print('  PSR < 0: Model SUSPENSION triggered')
    elif psr < 0.5: print('  PSR < 0.5: Mandatory model REVIEW triggered')
    else:           print('  PSR >= 0.5: Model performance within acceptable range')

In [ ]:
# ── Visualise Station 4a: Strategy Performance ──

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'DDF Station 4: Backtest Results — {focus} Signal-Driven Strategy', 
             fontsize=14, fontweight='bold')

# Cumulative returns
strat_cum = (1 + strat_rets).cumprod()
bh_cum    = (1 + bh_rets).cumprod()
axes[0].plot(strat_cum.index, strat_cum, color='#003366', linewidth=2, label='AI Strategy')
axes[0].plot(bh_cum.index,   bh_cum,   color='#C8102E', linewidth=1.5, linestyle='--', label='Buy & Hold')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Signal position
axes[1].fill_between(position.index, 0, position, step='post',
                     color='#003366', alpha=0.5, label='Long Position')
axes[1].set_ylabel('Position (1=Long, 0=Cash)')
axes[1].set_ylim(-0.1, 1.3)
axes[1].legend()
axes[1].grid(alpha=0.3)

# Drawdown
strat_cum_dd = (1 + strat_rets).cumprod()
roll_max     = strat_cum_dd.cummax()
drawdown     = (strat_cum_dd - roll_max) / roll_max
axes[2].fill_between(drawdown.index, drawdown, 0, color='#C8102E', alpha=0.4, label='Strategy Drawdown')
axes[2].set_ylabel('Drawdown')
axes[2].set_ylim(min(drawdown.min() * 1.3, -0.01), 0.02)
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Pipeline A complete.')

---
## Pipeline B — Long-Term ESG-Aware AI Investment Portfolio

### DDF Station 1: ETL — Fundamental Data and ESG Scores

Pipeline B operates at the long-horizon investment layer, using fundamental financial data
and ESG scores rather than technical indicators. In a production system, this data would
be sourced from quarterly ASX company announcements (XBRL-tagged financials), MSCI ESG
ratings, and Sustainalytics data feeds — with point-in-time versioning to prevent
look-ahead bias in backtests [BCBS 239, Basel Committee, 2013].

In [ ]:
# ── DDF Station 1b: ETL — Fundamental and ESG Data ──

fundamental_data = pd.DataFrame({
    'Ticker':        ['CBA.AX', 'BHP.AX', 'CSL.AX', 'RIO.AX', 'WES.AX',
                      'NAB.AX', 'TLS.AX', 'TCL.AX', 'MQG.AX', 'FMG.AX'],
    'Sector':        ['Financials', 'Materials', 'Healthcare', 'Materials', 'Consumer',
                      'Financials', 'Telecom', 'Utilities', 'Financials', 'Materials'],
    'PE_Ratio':      [15.2, 11.4, 38.7, 9.8, 22.3, 14.6, 20.1, 18.9, 16.8, 7.4],
    'ROE_pct':       [13.8, 18.2, 22.6, 20.1, 15.4, 12.9, 8.7, 10.2, 17.3, 26.8],
    'Debt_Equity':   [1.42, 0.38, 0.29, 0.35, 0.55, 1.28, 0.94, 2.11, 0.88, 0.62],
    'Div_Yield_pct': [4.8, 5.2, 1.4, 6.1, 3.8, 5.5, 4.2, 5.0, 3.1, 9.4],
    'ESG_Score':     [0.82, 0.54, 0.76, 0.61, 0.67, 0.79, 0.72, 0.80, 0.71, 0.38],
    'Market_Cap_B':  [182, 145, 138, 52, 58, 94, 44, 28, 85, 47],
}).set_index('Ticker')

print('Station 1 — Fundamental and ESG Data (10 ASX 200 stocks)')
print(fundamental_data.to_string())
print(f'\nESG Score range: {fundamental_data["ESG_Score"].min():.2f} '
      f'(FMG.AX — high emissions) to {fundamental_data["ESG_Score"].max():.2f} (CBA.AX)')

### DDF Station 2: Feature Engineering — Portfolio Factor Construction

### DDF Station 3: Model Design — MVO + ESG Optimisation (Eq 8.5)

The ESG-adjusted portfolio allocation solves:

$$\min_{\mathbf{w}} \; \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} - \lambda \, \mathbf{w}^\top \boldsymbol{\mu} - \kappa \, \mathbf{w}^\top \mathbf{e} \tag{8.5}$$

subject to $\mathbf{1}^\top \mathbf{w} = 1$, $\mathbf{w} \geq 0$, $w_i \leq 0.20$ (concentration limit)

**Parameters:**
- $\lambda$ = risk-aversion parameter (higher = more conservative)
- $\kappa$ = ESG preference parameter (higher = stronger ESG tilt)
- Concentration limit: max 20% per stock (simulating APRA superannuation diversification requirement)

In [ ]:
# ── DDF Stations 2b + 3b: MVO + ESG Portfolio Optimisation (Eq 8.5) ──

# Estimate expected returns from simulated price data (annualised log returns)
log_returns = np.log(prices).diff().dropna()
mu_vec  = log_returns.mean().values * 252          # annualised expected returns
Sigma   = log_returns.cov().values * 252            # annualised covariance matrix
esg_vec = fundamental_data['ESG_Score'].values      # ESG score vector e
n_assets = len(mu_vec)

def mvo_esg_objective(w, Sigma, mu, esg, lam, kap):
    portfolio_var    = w @ Sigma @ w
    expected_return  = mu @ w
    esg_contribution = esg @ w
    return portfolio_var - lam * expected_return - kap * esg_contribution

def optimise_portfolio(Sigma, mu, esg, lam=2.0, kap=0.0, max_weight=0.20):
    w0 = np.ones(n_assets) / n_assets
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0.0, max_weight)] * n_assets
    result = minimize(
        mvo_esg_objective, w0,
        args=(Sigma, mu, esg, lam, kap),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-10, 'maxiter': 1000}
    )
    return result.x if result.success else w0

# ── Solve for three scenarios ──
scenarios = {
    'Standard MVO\n(λ=2, κ=0)':          optimise_portfolio(Sigma, mu_vec, esg_vec, lam=2.0, kap=0.0),
    'ESG-Aware\n(λ=2, κ=0.5)':           optimise_portfolio(Sigma, mu_vec, esg_vec, lam=2.0, kap=0.5),
    'High ESG Preference\n(λ=2, κ=1.5)': optimise_portfolio(Sigma, mu_vec, esg_vec, lam=2.0, kap=1.5),
}

tickers = list(ASX_STOCKS.keys())
weight_df = pd.DataFrame(scenarios, index=tickers)

# ── Compute portfolio statistics for each scenario ──
def portfolio_stats(w, mu, Sigma, rf=RBA_CASH_RATE):
    ret = mu @ w
    vol = np.sqrt(w @ Sigma @ w)
    sr  = (ret - rf) / vol
    esg = esg_vec @ w
    return ret, vol, sr, esg

print('Station 3 — MVO+ESG Portfolio Weights (%)')
print('='*65)
print(f'{"Ticker":<10} {"Standard MVO":>14} {"ESG-Aware":>12} {"High ESG":>12} {"ESG Score":>10}')
print('-'*65)
for ticker in tickers:
    w1, w2, w3 = [scenarios[k][tickers.index(ticker)] for k in scenarios]
    esg = fundamental_data.loc[ticker, 'ESG_Score']
    print(f'{ticker:<10} {w1*100:>13.1f}% {w2*100:>11.1f}% {w3*100:>11.1f}% {esg:>10.2f}')
print('='*65)
for name, w in scenarios.items():
    ret, vol, sr, esg = portfolio_stats(w, mu_vec, Sigma)
    print(f'{name.replace(chr(10), " "):<35} SR={sr:.3f}  Ret={ret:.1%}  Vol={vol:.1%}  ESG={esg:.3f}')

In [ ]:
# ── Visualise MVO+ESG Optimisation Results ──

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('DDF Station 3: MVO + ESG Portfolio Optimisation (Eq 8.5)', 
             fontsize=14, fontweight='bold')

# Portfolio weights bar chart
x = np.arange(n_assets)
width = 0.28
colors = ['#003366', '#4472C4', '#C8102E']
scenario_labels = ['Standard MVO (κ=0)', 'ESG-Aware (κ=0.5)', 'High ESG (κ=1.5)']
for i, (name, w) in enumerate(scenarios.items()):
    axes[0].bar(x + i * width, w * 100, width, label=scenario_labels[i],
                color=colors[i], alpha=0.85)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(tickers, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Portfolio Weight (%)')
axes[0].set_title('Portfolio Weights by Scenario')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(20, color='red', linestyle='--', alpha=0.6, label='20% limit')

# ESG score overlay
esg_scores = fundamental_data['ESG_Score'].values
ax2 = axes[0].twinx()
ax2.scatter(x + width, esg_scores, color='orange', zorder=5, s=60, marker='D', label='ESG Score')
ax2.set_ylabel('ESG Score', color='orange')
ax2.set_ylim(0, 1.2)
ax2.tick_params(axis='y', labelcolor='orange')

# ESG score vs. weight change scatter
w_base  = list(scenarios.values())[0]
w_high  = list(scenarios.values())[2]
w_delta = (w_high - w_base) * 100
scatter_colors = ['#C8102E' if d < 0 else '#003366' for d in w_delta]
axes[1].scatter(esg_scores, w_delta, c=scatter_colors, s=100, zorder=5)
for i, ticker in enumerate(tickers):
    axes[1].annotate(ticker, (esg_scores[i], w_delta[i]),
                     textcoords='offset points', xytext=(6, 4), fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axvline(esg_scores.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'Mean ESG ({esg_scores.mean():.2f})')
axes[1].set_xlabel('ESG Score')
axes[1].set_ylabel('Weight Change: κ=0 → κ=1.5 (pp)')
axes[1].set_title('ESG Score vs. Weight Change from ESG Overlay')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('ESG insight: FMG.AX (ESG=0.38) weight falls significantly under high ESG preference;'
      ' CBA.AX (ESG=0.82) and TCL.AX (ESG=0.80) weights increase.')

### DDF Station 3c: AI Portfolio Narrative Generation

The Portfolio AI Agent generates a professional investment commentary synthesising the
quantitative optimisation results with qualitative market context. This implements the
AI Analyst framework [Fons et al., 2025], producing output annotated with the DR/FI/EK
segment classification.

In [ ]:
# ── DDF Station 3c: AI Portfolio Narrative (Gemini / Mock) ──

esg_aware_weights = list(scenarios.values())[1]
ret_b, vol_b, sr_b, esg_b = portfolio_stats(esg_aware_weights, mu_vec, Sigma)
top3_indices = esg_aware_weights.argsort()[-3:][::-1]
top3 = [(tickers[i], esg_aware_weights[i]) for i in top3_indices]

def build_portfolio_prompt(weights, tickers, stats):
    ret, vol, sr, esg = stats
    top = sorted(zip(tickers, weights), key=lambda x: -x[1])[:3]
    top_str = ', '.join([f'{t} ({w*100:.1f}%)' for t, w in top])
    return f"""
You are a senior portfolio manager at an Australian superannuation fund.
Write a 150-word investment commentary for the quarterly report based on these portfolio analytics:

Portfolio: ESG-aware MVO allocation (κ=0.5, λ=2)
Expected return: {ret:.1%} | Volatility: {vol:.1%} | Sharpe ratio: {sr:.2f} | Avg ESG score: {esg:.2f}
Top three positions: {top_str}
Benchmark: ASX 200 (S&P/ASX 200 Total Return)

The commentary should address: portfolio positioning rationale, ESG integration approach,
key risks (including regime change and model limitations), and outlook.
Use professional investment management language appropriate for superannuation beneficiaries.
""".strip()

def mock_portfolio_narrative(weights, tickers, stats):
    ret, vol, sr, esg = stats
    top = sorted(zip(tickers, weights), key=lambda x: -x[1])[:3]
    top_str = ', '.join([f'{t} ({w*100:.1f}%)' for t, w in top])
    return f"""
PORTFOLIO COMMENTARY — Q4 2023 | ESG-Aware ASX Equity Portfolio

[DR] The portfolio delivered an expected annualised return of {ret:.1%} with volatility
of {vol:.1%} per annum, yielding a Sharpe ratio of {sr:.2f} relative to the RBA cash
rate of 4.35%. The weighted average ESG score of {esg:.2f} reflects the portfolio's
responsible investment mandate.

[FI] Positioning is concentrated in the highest-quality, highest-ESG-scoring names:
{top_str}. This reflects the dual objective of risk-adjusted return maximisation and ESG
integration consistent with the fund's ASIC RG 97-compliant responsible investment policy.
The ESG overlay (κ=0.5) reduced allocation to high-emission materials stocks — notably
FMG.AX — while increasing weights in financials and utilities with strong governance scores.

[EK] The portfolio remains exposed to macroeconomic regime risk: the RBA's monetary policy
trajectory and global commodity demand from China are principal uncertainties. The
Performance Stability Ratio (Eq 8.6) is monitored monthly; a PSR breach below 0.5 would
trigger mandatory model review under the fund's SR 11-7-aligned governance framework.

[Segment annotations: DR=Direct Reference, FI=Financial Interpretation, EK=External Knowledge
 — classification per Fons et al. (2025) AI Analyst framework]
[Mock simulation — add GOOGLE_API_KEY to Colab Secrets for live Gemini narrative]
""".strip()

stats = (ret_b, vol_b, sr_b, esg_b)

if USE_LIVE_API:
    try:
        client = genai.GenerativeModel(GEMINI_MODEL)
        prompt = build_portfolio_prompt(esg_aware_weights, tickers, stats)
        response = client.generate_content(prompt)
        narrative = response.text
        print('Live Gemini portfolio narrative generated.')
    except Exception as e:
        print(f'Gemini API error ({e}); falling back to mock.')
        narrative = mock_portfolio_narrative(esg_aware_weights, tickers, stats)
else:
    narrative = mock_portfolio_narrative(esg_aware_weights, tickers, stats)

print()
print(narrative)

### DDF Station 4: Implementation — Portfolio Reporting and HMM Regime Detection

The final station produces investment-ready outputs: a performance attribution table,
ESG compliance report, and a Hidden Markov Model regime detector (Eq 8.7) that
monitors the market state and would trigger the Portfolio Agent's defensive response
protocol in a live system.

In [ ]:
# ── DDF Station 4b: Performance Reporting + HMM Regime Detection (Eq 8.7) ──

# ── Portfolio Performance Attribution ──
print('Station 4 — Portfolio Performance Attribution')
print('='*68)
print(f'{"Scenario":<30} {"Ret":>8} {"Vol":>8} {"SR":>8} {"ESG":>8} {"PSR*":>8}')
print('-'*68)
for name, w in scenarios.items():
    ret, vol, sr, esg = portfolio_stats(w, mu_vec, Sigma)
    # Simplified PSR: ratio of second-half to first-half SR from simulated returns
    sim_rets = log_returns @ w
    mid = len(sim_rets) // 2
    sr1 = (sim_rets.iloc[:mid].mean() * 252 - RBA_CASH_RATE) / (sim_rets.iloc[:mid].std() * np.sqrt(252) + 1e-9)
    sr2 = (sim_rets.iloc[mid:].mean() * 252 - RBA_CASH_RATE) / (sim_rets.iloc[mid:].std() * np.sqrt(252) + 1e-9)
    psr_val = sr2 / sr1 if abs(sr1) > 0.01 else float('nan')
    label = name.replace('\n', ' ')
    print(f'{label:<30} {ret:>7.1%} {vol:>8.1%} {sr:>7.2f} {esg:>8.3f} {psr_val:>8.3f}')
print('='*68)
print('* PSR: Performance Stability Ratio (Eq 8.6) — first vs. second simulation half')

# ── HMM Regime Detection (Eq 8.7) ──
print()
print('Regime Detection — Hidden Markov Model (Eq 8.7, K=2)')
print('='*52)

try:
    from hmmlearn.hmm import GaussianHMM
    asx200_rets = log_returns.mean(axis=1).values.reshape(-1, 1)
    hmm = GaussianHMM(n_components=2, covariance_type='full', n_iter=100, random_state=42)
    hmm.fit(asx200_rets)
    regimes = hmm.predict(asx200_rets)
    # Label regimes: 0 = low-vol, 1 = high-vol (by volatility of each state's returns)
    r0_vol = asx200_rets[regimes == 0].std()
    r1_vol = asx200_rets[regimes == 1].std()
    low_vol_state  = 0 if r0_vol < r1_vol else 1
    high_vol_state = 1 - low_vol_state
    regime_labels  = np.where(regimes == low_vol_state, 'Low-Vol', 'High-Vol')
    A = hmm.transmat_
    print(f'Transition matrix A (Eq 8.7):')
    print(f'  A[Low→Low]:   {A[low_vol_state, low_vol_state]:.3f}  |  A[Low→High]: {A[low_vol_state, high_vol_state]:.3f}')
    print(f'  A[High→Low]:  {A[high_vol_state, low_vol_state]:.3f}  |  A[High→High]: {A[high_vol_state, high_vol_state]:.3f}')
    current_regime = regime_labels[-1]
    p_high_next    = A[low_vol_state, high_vol_state] if current_regime == 'Low-Vol' \
                     else A[high_vol_state, high_vol_state]
    print(f'\nCurrent detected regime: {current_regime}')
    print(f'P(High-Vol next period):  {p_high_next:.3f}')
    if current_regime == 'High-Vol' or p_high_next > 0.35:
        print('ACTION: Defensive rotation protocol activated (reduce leverage, shift to low-beta)')
    else:
        print('ACTION: Maintain current allocation; continue monitoring')
    low_pct  = (regime_labels == 'Low-Vol').mean()
    high_pct = (regime_labels == 'High-Vol').mean()
    print(f'\nRegime distribution: Low-Vol {low_pct:.1%} | High-Vol {high_pct:.1%}')
except ImportError:
    print('hmmlearn not installed — running simplified volatility-based regime indicator.')
    roll_vol = log_returns.mean(axis=1).rolling(20).std() * np.sqrt(252)
    vol_threshold = roll_vol.median()
    regime_simple = np.where(roll_vol > vol_threshold, 'High-Vol', 'Low-Vol')
    current = regime_simple[-1] if not pd.isna(roll_vol.iloc[-1]) else 'Unknown'
    print(f'Simplified regime (rolling 20-day vol vs. median): current = {current}')

In [ ]:
# ── Visualise Station 4b: Portfolio Attribution + Regime Detection ──

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
fig.suptitle('DDF Station 4: Portfolio Reporting Dashboard', fontsize=14, fontweight='bold')

# ── Efficient frontier (vary lambda) ──
ax1 = fig.add_subplot(gs[0, 0])
lambdas = np.linspace(0.1, 8, 40)
frontier_vols, frontier_rets, frontier_esgs = [], [], []
for lam in lambdas:
    w = optimise_portfolio(Sigma, mu_vec, esg_vec, lam=lam, kap=0.0)
    r, v, _, e = portfolio_stats(w, mu_vec, Sigma)
    frontier_vols.append(v)
    frontier_rets.append(r)
    frontier_esgs.append(e)
sc = ax1.scatter(frontier_vols, frontier_rets, c=frontier_esgs, cmap='RdYlGn',
                 s=40, vmin=0.5, vmax=0.8)
plt.colorbar(sc, ax=ax1, label='Avg ESG Score')
for name, w in scenarios.items():
    r, v, _, _ = portfolio_stats(w, mu_vec, Sigma)
    ax1.scatter([v], [r], s=120, marker='*', zorder=5)
    ax1.annotate(name.split('\n')[0], (v, r), fontsize=7, xytext=(5, 3),
                 textcoords='offset points')
ax1.set_xlabel('Portfolio Volatility (p.a.)')
ax1.set_ylabel('Expected Return (p.a.)')
ax1.set_title('Efficient Frontier (colour = ESG score)')
ax1.grid(alpha=0.3)

# ── ESG score vs. final weight ──
ax2 = fig.add_subplot(gs[0, 1])
esg_aware_w = list(scenarios.values())[1]
high_esg_w  = list(scenarios.values())[2]
ax2.scatter(esg_vec, esg_aware_w * 100, color='#003366', s=80, label='κ=0.5', zorder=5)
ax2.scatter(esg_vec, high_esg_w  * 100, color='#C8102E', s=80, marker='^', label='κ=1.5', zorder=5)
for i, t in enumerate(tickers):
    ax2.annotate(t, (esg_vec[i], esg_aware_w[i]*100), fontsize=7, xytext=(4, 2),
                 textcoords='offset points')
ax2.set_xlabel('ESG Score')
ax2.set_ylabel('Portfolio Weight (%)')
ax2.set_title('ESG Score vs. Portfolio Weight')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# ── HMM regime detection timeline ──
ax3 = fig.add_subplot(gs[1, :])
try:
    asx_series = log_returns.mean(axis=1)
    colors_regime = ['#003366' if r == low_vol_state else '#C8102E' for r in regimes]
    ax3.bar(dates[1:], asx_series.values, color=colors_regime, alpha=0.7, width=1)
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#003366', alpha=0.7, label='Low-Vol Regime (Eq 8.7: state 0)'),
        Patch(facecolor='#C8102E', alpha=0.7, label='High-Vol Regime (Eq 8.7: state 1)'),
    ]
    ax3.legend(handles=legend_elements, fontsize=9)
    ax3.set_title('HMM Regime Detection — Equal-Weighted Portfolio Daily Returns')
    ax3.set_ylabel('Log Return')
    ax3.grid(alpha=0.3)
except NameError:
    roll_vol_plot = log_returns.mean(axis=1).rolling(20).std() * np.sqrt(252)
    ax3.plot(dates[1:], roll_vol_plot.values, color='#003366', linewidth=1.5)
    ax3.axhline(roll_vol_plot.median(), color='#C8102E', linestyle='--', label='Median vol threshold')
    ax3.set_ylabel('Rolling 20-day Annualised Volatility')
    ax3.set_title('Simplified Regime Indicator (Rolling Volatility)')
    ax3.legend(fontsize=9)
    ax3.grid(alpha=0.3)

plt.show()
print('Pipeline B complete. Both pipelines operational.')

---
## Extension: Streamlit Interactive Dashboard

The code below shows the structure of a Streamlit application that wraps both pipelines
into an interactive interface. Save as `trading_app.py` and run with `streamlit run trading_app.py`.

Parameters adjustable via sidebar sliders:
- Pipeline A: RSI window, MACD fast/slow/signal, Bollinger window/std
- Pipeline B: Risk-aversion λ, ESG preference κ, concentration limit, benchmark
- Regime detection: HMM components K, rolling window for simplified detector

**Educational note:** This dashboard maps directly to DDF Station 4 — the Implementation
layer where model outputs are delivered to human decision-makers via an interface [Hric & Lin, 2026].

In [ ]:
# ── Extension: Streamlit Dashboard (save as trading_app.py) ──
# Run with: streamlit run trading_app.py

STREAMLIT_CODE = '''
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

st.set_page_config(page_title="FINS5557 Week 8 — ASX AI Trading Platform",
                   layout="wide", page_icon="📈")
st.title("FINS5557 Week 8: ASX AI Trading & Investment Platform")
st.caption("Hric & Lin, 2026 — UNSW Business School | Educational Simulation")

tab1, tab2 = st.tabs(["Pipeline A: Signal Trading", "Pipeline B: ESG Portfolio"])

with st.sidebar:
    st.header("Parameters")
    st.subheader("Pipeline A")
    rsi_window = st.slider("RSI Window", 7, 28, 14)
    macd_fast  = st.slider("MACD Fast", 6, 20, 12)
    macd_slow  = st.slider("MACD Slow", 20, 40, 26)
    st.subheader("Pipeline B")
    lam = st.slider("Risk Aversion λ", 0.5, 5.0, 2.0, 0.5)
    kap = st.slider("ESG Preference κ", 0.0, 2.0, 0.5, 0.25)
    max_w = st.slider("Max Weight (%)", 10, 40, 20) / 100

with tab1:
    st.subheader("Signal-Driven ASX Equity Trading")
    st.info("Technical indicator signals for CBA.AX | Adjust parameters in sidebar")
    st.write(f"RSI window: {rsi_window} | MACD: {macd_fast}/{macd_slow}")
    st.write("**DDF Station 4 outputs:** Sharpe Ratio, MDD, Cumulative Return, PSR")
    st.code("# Full implementation: see week8-case-study.ipynb", language="python")

with tab2:
    st.subheader("ESG-Aware MVO Portfolio (Eq 8.5)")
    st.latex(r"""
        \min_{\mathbf{w}} \; \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}
        - \lambda \, \mathbf{w}^\top \boldsymbol{\mu}
        - \kappa \, \mathbf{w}^\top \mathbf{e}
    """)
    col1, col2, col3 = st.columns(3)
    col1.metric("Risk Aversion λ", f"{lam:.1f}")
    col2.metric("ESG Preference κ", f"{kap:.2f}")
    col3.metric("Max Weight", f"{max_w*100:.0f}%")
    st.info("ESG overlay: κ=0 → standard MVO | κ>0 → ESG-tilted allocation")
    st.write("**ASIC RG 97 disclosure:** ESG methodology and κ calibration must be disclosed to beneficiaries")
    st.code("# Full implementation: see week8-case-study.ipynb", language="python")

st.sidebar.divider()
st.sidebar.caption("FINS5557 | Week 8 | Hric & Lin, 2026")
'''

print('Streamlit app structure:')
print('  - Two tabs: Pipeline A (signal trading) | Pipeline B (ESG portfolio)')
print('  - Sidebar sliders: RSI window, MACD params, λ, κ, max weight')
print('  - Displays MVO formula (Eq 8.5) with live parameter values')
print('  - ASIC RG 97 disclosure reminder embedded in UI')
print()
print('To run the dashboard:')
print('  1. Save the STREAMLIT_CODE string above to trading_app.py')
print('  2. Install: pip install streamlit')
print('  3. Run: streamlit run trading_app.py')
print()
print('Week 8 case study complete.')
print('Both pipelines operational — mock simulation runs without any API key.')
print('Covers: ETL → Feature Engineering → Model Design → Implementation (DDF Stations 1–4).')